# Circadiaan ML — Stemming & Stress Voorspelling

**Project R.E.M. · ML-notebook**

Voorspelling van `mood_delta` en `stress_delta` op basis van circadiaan afwijkingen en sessiefuncties.
Modellen: Ridge, Random Forest, Gradient Boosting (+ DummyMean als nulbaseline).
Validatie: leave-one-session-out kruisvalidatie met imputatie per fold om lekkage te vermijden.

**REUSE_MODEL = True** → laad opgeslagen modellen uit `models/circadian_ml/`.  
**REUSE_MODEL = False** → train alle modellen opnieuw en sla op.

In [25]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
# ── Configuratie ─────────────────────────────────────────────────────────────
# PARTICIPANTS: "all" → auto-detecteer vanuit feature_matrix.csv
#               lijst  → bijv. ["kokosnoot", "peer"]
PARTICIPANTS = "all"

# REUSE_MODEL: True  → laad opgeslagen modellen (snel)
#              False → train opnieuw en sla op
REUSE_MODEL = True

# Features (hrv_rmssd en avg_resp_daily meegerekend maar sterk NaN)
FEATURE_COLS = [
    "baseline_deviation_entry",
    "hr_baseline_deviation",
    "hour_of_day",
    "day_of_week",
    "playlist_calm",
    "playlist_energy",
    "mood_before_score",
    "bb_start",
    "days_since_last_session",
    "during_stress_mean",
    "post_stress_mean",
    "during_hr_mean",
    "post_hr_mean",
    "pre_state_encoded",
    "hrv_rmssd",
    "avg_resp_daily",
]

TARGET_COLS = {"mood_delta": "Stemming Delta", "stress_delta": "Stress Delta"}

MODELS = {
    "DummyMean":        DummyRegressor(strategy="mean"),
    "Ridge":            Ridge(alpha=1.0),
    "RandomForest":     RandomForestRegressor(n_estimators=100, max_depth=3, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=50, max_depth=2, learning_rate=0.1, random_state=42
    ),
}

In [27]:
# ── Paden ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT        = Path().resolve().parent.parent
DATA_ROOT           = PROJECT_ROOT / "data"
COMBINED_DIR        = DATA_ROOT / "analysis" / "circadian_baselines"
MODELS_DIR          = PROJECT_ROOT / "models" / "circadian_ml"
FEATURE_MATRIX_PATH = COMBINED_DIR / "feature_matrix.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Donker thema ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1218",
    "axes.facecolor":   "#181e2a",
    "axes.edgecolor":   "#4a5568",
    "axes.labelcolor":  "#e2e8f0",
    "xtick.color":      "#a0aec0",
    "ytick.color":      "#a0aec0",
    "text.color":       "#e2e8f0",
    "grid.color":       "#2d3748",
    "grid.alpha":       0.4,
    "legend.facecolor": "#1a2035",
    "legend.edgecolor": "#4a5568",
    "figure.dpi":       120,
    "font.family":      "monospace",
})

OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]

## Hulpfuncties

In [28]:
def prepare_data(
    feature_matrix_path: Path,
    target: str,
    participants: list[str],
) -> tuple[pd.DataFrame, pd.Series, list[str], pd.Series]:
    """Laad feature matrix, filter deelnemers, verwijder NaN-doelrijen.
    Imputatie gebeurt NIET hier — dat vindt per fold plaats in de Pipeline.
    """
    fm = pd.read_csv(feature_matrix_path)
    fm = fm[fm["participant"].isin(participants)].reset_index(drop=True)
    fm = fm.dropna(subset=[target]).reset_index(drop=True)

    participant_dummies = pd.get_dummies(fm["participant"], prefix="p", drop_first=True)
    feature_cols = FEATURE_COLS + list(participant_dummies.columns)
    X = pd.concat([fm[FEATURE_COLS], participant_dummies], axis=1)
    y = fm[target]
    groups = fm["participant"]
    return X, y, feature_cols, groups


def _fit_pipeline(X: pd.DataFrame, y: pd.Series, model) -> Pipeline:
    """Fit een imputatie + model pipeline op de volledige dataset."""
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model",   clone(model)),
    ])
    pipe.fit(X, y)
    return pipe

In [29]:
def run_loo_cv(X: pd.DataFrame, y: pd.Series, model) -> dict:
    """Leave-one-session-out CV met imputatie per fold (geen lekkage).
    Geeft MAE, RMSE, R², overfitting-gap en LOO-voorspellingen terug.
    """
    loo = LeaveOneOut()
    y_pred = np.full(len(y), np.nan)
    train_scores = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model",   clone(model)),
        ])
        pipe.fit(X_train, y_train)
        y_pred[test_idx] = pipe.predict(X_test)
        train_scores.append(pipe.score(X_train, y_train))

    mae           = mean_absolute_error(y, y_pred)
    rmse          = float(np.sqrt(mean_squared_error(y, y_pred)))
    r2            = r2_score(y, y_pred)
    train_r2_mean = float(np.mean(train_scores))

    return {
        "y_pred":        y_pred,
        "MAE":           mae,
        "RMSE":          rmse,
        "R2_LOO":        r2,
        "R2_train_mean": train_r2_mean,
        "overfit_gap":   train_r2_mean - r2,
    }


def train_and_evaluate(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    target_name: str,
) -> tuple[pd.DataFrame, dict]:
    """Voer LOO-CV uit voor alle modellen.
    Geeft vergelijkings-DataFrame en per-model-resultaten terug.
    """
    results = {}
    rows    = []

    for name, model in MODELS.items():
        res = run_loo_cv(X, y, model)
        res["model_name"] = name
        results[name] = res

        row = {
            "model":         name,
            "MAE":           res["MAE"],
            "RMSE":          res["RMSE"],
            "R2_LOO":        res["R2_LOO"],
            "R2_train_mean": res["R2_train_mean"],
            "overfit_gap":   res["overfit_gap"],
        }

        # Per-deelnemer MAE
        for p in groups.unique():
            mask = groups == p
            if mask.sum() > 0:
                row[f"MAE_{p}"] = mean_absolute_error(y[mask], res["y_pred"][mask])

        rows.append(row)

    return pd.DataFrame(rows), results

## Data laden

In [30]:
if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"Feature matrix niet gevonden: {FEATURE_MATRIX_PATH}\n"
        "Voer eerst scripts/analysis/circadian_baseline.py uit."
    )

fm_raw = pd.read_csv(FEATURE_MATRIX_PATH)

# Auto-detecteer deelnemers
if PARTICIPANTS == "all":
    participants = sorted(fm_raw["participant"].unique().tolist())
else:
    participants = list(PARTICIPANTS)

print(f"Feature matrix geladen: {len(fm_raw)} sessies")
print(f"Deelnemers ({len(participants)}): {participants}")
print(f"Sessies per deelnemer:")
display(fm_raw["participant"].value_counts().to_frame())

Feature matrix geladen: 82 sessies
Deelnemers (4): ['bosbes', 'kokosnoot', 'limoen', 'peer']
Sessies per deelnemer:


,count
participant,
kokosnoot,40
peer,30
limoen,8
bosbes,4


## Model trainen / laden

Alle vier modellen draaien LOO-CV. Imputatie vindt per fold plaats zodat er geen lekkage is.
Naast de gefitte pipelines worden ook de volledige datasets opgeslagen — de visualisatienotebook
(`circadian_ml_viz.ipynb`) heeft die nodig voor SHAP en permutation importance.

In [31]:
models_path = MODELS_DIR / "models.pkl"

if REUSE_MODEL:
    if not models_path.exists():
        raise FileNotFoundError(
            f"Opgeslagen modellen niet gevonden: {models_path}\n"
            "Zet REUSE_MODEL = False om de modellen opnieuw te trainen."
        )
    with open(models_path, "rb") as f:
        saved = pickle.load(f)
    all_comparisons = saved["comparisons"]
    all_results     = saved["results"]
    all_fitted      = saved["fitted"]
    all_X           = saved["X_data"]
    all_y           = saved["y_data"]
    all_groups      = saved["groups"]
    feature_names   = saved["feature_names"]
    print(f"Modellen geladen vanuit {models_path}")

else:
    all_comparisons = {}
    all_results     = {}
    all_fitted      = {}
    all_X           = {}
    all_y           = {}
    all_groups      = {}
    feature_names   = None

    for target, label in TARGET_COLS.items():
        print(f"\n{'─'*50}")
        print(f"Target: {label} ({target})")

        X, y, feat_names, groups = prepare_data(FEATURE_MATRIX_PATH, target, participants)
        feature_names = feat_names  # zelfde voor beide targets

        print(f"  Sessies: {len(y)} | Features: {len(feat_names)}")
        print(f"  Sessies per deelnemer: {dict(groups.value_counts())}")

        comparison, results = train_and_evaluate(X, y, groups, target)
        fitted = {name: _fit_pipeline(X, y, model) for name, model in MODELS.items()}

        all_comparisons[target] = comparison
        all_results[target]     = results
        all_fitted[target]      = fitted
        all_X[target]           = X
        all_y[target]           = y
        all_groups[target]      = groups

        # Sla resultaat-CSV op
        COMBINED_DIR.mkdir(parents=True, exist_ok=True)
        comparison.to_csv(COMBINED_DIR / f"model_results_{target}.csv", index=False)
        print(f"  Opgeslagen: model_results_{target}.csv")

    # Sla modellen en data op
    with open(models_path, "wb") as f:
        pickle.dump({
            "comparisons":  all_comparisons,
            "results":      all_results,
            "fitted":       all_fitted,
            "X_data":       all_X,
            "y_data":       all_y,
            "groups":       all_groups,
            "feature_names": feature_names,
        }, f)
    print(f"\nModellen opgeslagen: {models_path}")


──────────────────────────────────────────────────
Target: Stemming Delta (mood_delta)
  Sessies: 82 | Features: 19
  Sessies per deelnemer: {'kokosnoot': np.int64(40), 'peer': np.int64(30), 'limoen': np.int64(8), 'bosbes': np.int64(4)}
  Opgeslagen: model_results_mood_delta.csv

──────────────────────────────────────────────────
Target: Stress Delta (stress_delta)
  Sessies: 64 | Features: 19
  Sessies per deelnemer: {'kokosnoot': np.int64(34), 'peer': np.int64(25), 'bosbes': np.int64(4), 'limoen': np.int64(1)}
  Opgeslagen: model_results_stress_delta.csv

Modellen opgeslagen: C:\Users\astri\Desktop\Data_Scientist\Eindwerk\spotify-project\models\circadian_ml\models.pkl


## Resultaten

LOO-CV scores per model. Lagere MAE = beter. R² < 0 betekent slechter dan de gemiddelde voorspeller.
Overfitting-gap = R²_train − R²_LOO: groot verschil duidt op overfitting.

In [32]:
for target, label in TARGET_COLS.items():
    if target not in all_comparisons:
        print(f"Geen resultaten voor {target}")
        continue

    comparison = all_comparisons[target]
    y          = all_y[target]

    print(f"\n{'═'*60}")
    print(f"  {label.upper()} ({target})  ·  {len(y)} sessies")
    print(f"{'═'*60}")

    display(
        comparison[["model", "MAE", "RMSE", "R2_LOO", "R2_train_mean", "overfit_gap"]]
        .round(3)
    )

    # Overfitting-waarschuwingen
    for _, row in comparison.iterrows():
        if row["model"] != "DummyMean" and row["overfit_gap"] > 0.5:
            print(f"  ⚠ OVERFIT: {row['model']} — train R²={row['R2_train_mean']:.3f} vs LOO R²={row['R2_LOO']:.3f}")

    # Vergelijk beste model met DummyMean
    non_dummy = comparison[comparison["model"] != "DummyMean"]
    best_row  = non_dummy.loc[non_dummy["MAE"].idxmin()]
    dummy_mae = comparison.loc[comparison["model"] == "DummyMean", "MAE"].values[0]
    print(f"  Beste model: {best_row['model']} (MAE={best_row['MAE']:.3f})")
    if best_row["MAE"] >= dummy_mae:
        print(
            f"  ✗ Geen model verslaat DummyMean (MAE={dummy_mae:.3f}) — "
            f"features voorspellen {target} niet beter dan het gemiddelde (N={len(y)})."
        )
    else:
        gain = (dummy_mae - best_row["MAE"]) / dummy_mae * 100
        print(f"  ✓ {gain:.1f}% verbetering t.o.v. DummyMean")


════════════════════════════════════════════════════════════
  STEMMING DELTA (mood_delta)  ·  82 sessies
════════════════════════════════════════════════════════════


,model,MAE,RMSE,R2_LOO,R2_train_mean,overfit_gap
0,DummyMean,1.817,2.460,-0.025,0.000,0.025
1,Ridge,1.578,2.007,0.318,0.587,0.269
2,RandomForest,1.666,2.128,0.233,0.681,0.448
3,GradientBoosting,1.778,2.295,0.108,0.820,0.712


  ⚠ OVERFIT: GradientBoosting — train R²=0.820 vs LOO R²=0.108
  Beste model: Ridge (MAE=1.578)
  ✓ 13.2% verbetering t.o.v. DummyMean

════════════════════════════════════════════════════════════
  STRESS DELTA (stress_delta)  ·  64 sessies
════════════════════════════════════════════════════════════


,model,MAE,RMSE,R2_LOO,R2_train_mean,overfit_gap
0,DummyMean,9.649,12.883,-0.032,0.000,0.032
1,Ridge,2.866,4.571,0.870,0.960,0.090
2,RandomForest,6.437,9.094,0.486,0.839,0.354
3,GradientBoosting,6.230,8.678,0.532,0.954,0.422


  Beste model: Ridge (MAE=2.866)
  ✓ 70.3% verbetering t.o.v. DummyMean
